In [35]:
from openai import OpenAI
import requests
from pydantic_ai import Agent 
import os

from dotenv import load_dotenv

In [25]:

def get_page(title: str) -> str:
    """
    Retrieve the raw content of a Wikipedia page.

    Args:
        title (str): The exact title of the Wikipedia page.

    Returns:
        str: The page content as a string.
    """

    # Step 1: Define the Wikipedia Get Page API URL.
    url = "https://en.wikipedia.org/w/index.php"

    # Step 2: Specify the page title and request raw content.
    params = {
        "title": title,
        "action": "raw"
    }

    # Step 3: Define the required User-Agent header.
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }

    # Step 4: Send an HTTP GET request to Wikipedia.
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=15
    )

    # Step 5: Check whether the request succeeded.
    response.raise_for_status()

    # Step 6: Return the raw page content as a string.
    return response.text


In [19]:
 
def search_wikipedia(query: str) -> list[dict]:
    """
    Search Wikipedia for articles matching a query string.

    Args:
        query (str): The search term to look up on Wikipedia.

    Returns:
        list[dict]: A list of Wikipedia search result objects.
    """

    # Wikipedia Search API endpoint.
    url = "https://en.wikipedia.org/w/api.php"

    # API parameters.
    # requests will automatically encode spaces in the query.
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query
    }

    # Wikipedia requires a User-Agent header.
    headers = {
        "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
    }

    # Send a GET request to Wikipedia.
    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=15
    )

    # Raise an exception if the HTTP request failed.
    response.raise_for_status()

    # Convert the JSON response into a Python dictionary.
    data = response.json()

    # Extract the search results from query.search.
    search_results = data["query"]["search"]

    # Return the list of matching articles.
    return search_results


In [20]:

# Search Wikipedia for the word "capybara".
# Task1: Search Wikipedia for the word "capybara" and print the number of results and the first result.
search_results = search_wikipedia("capybara")

# Print the number of results returned by the API.
print("Number of search results:", len(search_results))

# Display the first search result.
print(search_results[0])


Number of search results: 10
{'ns': 0, 'title': 'Capybara', 'pageid': 6776, 'size': 44903, 'wordcount': 4312, 'snippet': 'The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South', 'timestamp': '2026-10-11T22:03:04Z'}


In [21]:

# Iterate through every search result.
for result in search_results:

    # Extract the title field from the result.
    title = result["title"]

    # Print the article title.
    print(title)


Capybara
Lesser capybara
Capybara (disambiguation)
Capybara Games
Capybara Party
Capybara (software)
Hydrochoerus
Caviidae
Yuzu bath
Flow (2024 film)


In [24]:


# Question 2: Search Wikipedia for "capybara" again.

# Step 1: Call our search_wikipedia function again.
search_results = search_wikipedia("capybara")

# Count all results.
print("Total returned:", len(search_results))

# Step 2: Create an empty list to store matching titles.
capybara_titles = []

# Step 3: Examine each result in the query.search array.
for result in search_results:

    # Extract the title of the Wikipedia article.
    title = result["title"]

    # Convert the title to lowercase for case-insensitive matching.
    if "capybara" in title.lower():

        # Add the matching title to our list.
        capybara_titles.append(title)

# Step 4: Print all matching titles.
print("Titles containing 'capybara':")

for title in capybara_titles:
    print(title)

# Step 5: Count the matching titles.
print("\nNumber of matching titles:", len(capybara_titles))



Total returned: 10
Titles containing 'capybara':
Capybara
Lesser capybara
Capybara (disambiguation)
Capybara Games
Capybara Party
Capybara (software)

Number of matching titles: 6


In [ ]:

# Step 1: Call get_page to fetch the Capybara article.
# This sends a new HTTP request to Wikipedia.
# Task 3 Define the Get Page Tool
content = get_page("Capybara")

# Step 2: Check the returned data type.
print("Content type:", type(content))

# Step 3: Count characters in the article.
character_count = len(content)

# Step 4: Print the number of characters.
print("Number of characters:", character_count)

# Step 5: Display the first 500 characters.
print("\nFirst 500 characters:")
print(content[:500])


Content type: <class 'str'>
Number of characters: 44767

First 500 characters:
{{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=''Hydrochoerus hydrochaeris'' |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en}}</ref>
| fossil_range = {{Fossilrange|Late Pleistocene|Present}}
| image = Capybaracropped.jpg
| image_alt = A capybara 


In [33]:

# Define instructions explaining how the agent should work.

instructions = """
You are a Wikipedia research assistant.

Answer user questions using information from Wikipedia.

Available tools:
1. search_wikipedia: Search Wikipedia for relevant pages.
2. get_page: Retrieve the full content of a Wikipedia page.

When answering a question about a topic:
- First, search Wikipedia for relevant articles.
- Then, retrieve the most relevant page using get_page.
- Use the retrieved information to formulate your answer.

When the user provides a specific Wikipedia page URL:
- Extract the page title from the URL.
- Use get_page to retrieve that page directly.

Provide clear and concise answers.

Use only facts found in the retrieved Wikipedia content.
If you cannot find the answer, inform the user.
"""


In [ ]:

# Load the environment variables from .env.
load_dotenv()

# Create a PydanticAI agent.

wikipedia_agent = Agent(

    # Name used to identify the agent.
    name="wikipedia",

    # OpenAI model used by the agent.
    model="openai:gpt-4o-mini",

    # Instructions controlling agent behaviour.
    instructions=instructions,

    # Register the two tools created in Questions 1 and 3.
    tools=[
        search_wikipedia,
        get_page
    ]
)

Wikipedia agent created successfully.


In [43]:

# Ask the Wikipedia agent a question.
# Task 4: Ask the Wikipedia agent a question and print the answer.

# Step 1: Define the question.
question = (
    "What is this page about? "
    "https://en.wikipedia.org/wiki/Capybara"
)

# Step 2: Run the Wikipedia agent.
agent_result = await wikipedia_agent.run(question)

# Step 3: Print the final answer.
print("AGENT ANSWER:")
print(agent_result.output)

# Step 4: Get the token usage information.
usage = agent_result.usage

# Step 5: Print token counts.
print("\nTOKEN USAGE:")
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

# Step 4: Verify that get_page was called.
print("\nTOOL CALLS:")

for message in agent_result.all_messages():
    for part in message.parts:
        if part.part_kind == "tool-call":
            print(part.tool_name, part.args)



AGENT ANSWER:
The Wikipedia page on **capybaras** describes them as the largest living rodents, scientifically known as *Hydrochoerus hydrochaeris*. They are native to most South American countries, except Chile, and are semiaquatic herbivores that typically inhabit savannas and dense forests near freshwater bodies. 

### Key Points:

- **Physical Description**: Capybaras have a barrel-shaped body, reddish-brown upper fur, and coarse hair ranging from 30 to 120 mm in length. They typically weigh between 35 to 66 kg and measure about 106 to 134 cm in length.

- **Social Behavior**: They are social animals that often live in groups ranging from 10 to 20 individuals, but they can form larger groups of up to 100, especially during dry seasons.

- **Diet**: Their diet mainly consists of grasses and aquatic plants, and they are known to practice coprophagy, consuming their own feces to maximize nutrient absorption.

- **Reproduction**: Capybaras have a gestation period of 130–150 days and co

In [44]:

# Step 1: Use the exact question from the homework.
question = "What are the main threats to capybara populations?"

# Step 2: Run the existing PydanticAI Wikipedia agent.
result_q6 = await wikipedia_agent.run(question)

# Step 3: Print the agent's final answer.
print("AGENT ANSWER:")
print(result_q6.output)

# Step 4: Display token usage.
print("\nTOKEN USAGE:")
print(result_q6.usage)


AGENT ANSWER:
The main threats to capybara populations include:

1. **Hunting**: Capybaras are hunted for their meat and hides in various regions of South America. The demand for capybara products leads to reductions in their local populations.

2. **Human Conflict**: In some areas, capybaras are seen as competition for livestock grazing, leading to them being killed by farmers.

3. **Habitat Destruction**: The destruction of wetland habitats for agriculture and urban development poses significant risks to their natural environments.

4. **Predation**: Although not a direct threat to their overall populations, capybaras are preyed upon by large predators such as jaguars, ocelots, and caimans.

Despite these threats, capybaras are not currently considered threatened as a species, and their populations are generally stable across most of their range, aided by their ability to breed rapidly and adapt to urbanization.

TOKEN USAGE:
RunUsage(cost=Decimal('0.00309105'), details={'reasoning_t

In [47]:

# Get all tool calls from the agent's message history.
tool_calls = [
    part
    for message in result_q6.all_messages()
    for part in message.parts
    if part.part_kind == "tool-call"
]

# Display tool calls in order.
print("TOOLS USED IN ORDER:")

for i, call in enumerate(tool_calls, start=1):
    print(f"{i}. {call.tool_name}({call.args})")

# Display the total number of tool calls.
print("\nTOTAL TOOL CALLS:", len(tool_calls))

# Display the final answer.
print("\nFINAL AGENT ANSWER:")
print(result_q6.output)


TOOLS USED IN ORDER:
1. search_wikipedia({"query":"Capybara threats"})
2. search_wikipedia({"query":"Capybara"})
3. get_page({"title":"Capybara"})

TOTAL TOOL CALLS: 3

FINAL AGENT ANSWER:
The main threats to capybara populations include:

1. **Hunting**: Capybaras are hunted for their meat and hides in various regions of South America. The demand for capybara products leads to reductions in their local populations.

2. **Human Conflict**: In some areas, capybaras are seen as competition for livestock grazing, leading to them being killed by farmers.

3. **Habitat Destruction**: The destruction of wetland habitats for agriculture and urban development poses significant risks to their natural environments.

4. **Predation**: Although not a direct threat to their overall populations, capybaras are preyed upon by large predators such as jaguars, ocelots, and caimans.

Despite these threats, capybaras are not currently considered threatened as a species, and their populations are generally